# Thai Road Safety — Model Training

3 ส่วนหลัก:
1. **EDA & Target Analysis** — ทำความเข้าใจข้อมูลและปัญหา Imbalanced Data
2. **Model Training** — เทียบ Baseline vs Final XGBoost
3. **AI Search Evaluation** — ทดสอบ Hybrid Semantic Search

## 1. โหลดข้อมูล

In [1]:
from data_prep import load_and_clean_data
import pandas as pd

df = load_and_clean_data()
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"หมวดหมู่ cause_clean: {df['cause_clean'].nunique()} หมวด")
print(f"ผลรวมแถวตรวจสอบ: {df['cause_clean'].value_counts().sum():,}")

Dataset: 151,778 rows, 20 columns
หมวดหมู่ cause_clean: 43 หมวด
ผลรวมแถวตรวจสอบ: 151,778


## 2. วิเคราะห์ Target — Imbalanced Data

In [2]:
print("สัดส่วน target (ตาย/ไม่ตาย):")
print(df['target'].value_counts(normalize=True))
print()
print("⚠️  ข้อมูลไม่สมดุล (Imbalanced): ไม่ตาย 88.3% vs ตาย 11.7%")
print("→ โมเดลที่ทาย 'ไม่ตาย' ทุกครั้งจะได้ Accuracy 88% ฟรีๆ โดยไม่เรียนรู้อะไร")
print("→ ต้องดู Recall ของ class 1 (ตาย) เป็นหลัก ไม่ใช่ Accuracy รวม")

สัดส่วน target (ตาย/ไม่ตาย):
target
0    0.883409
1    0.116591
Name: proportion, dtype: float64

⚠️  ข้อมูลไม่สมดุล (Imbalanced): ไม่ตาย 88.3% vs ตาย 11.7%
→ โมเดลที่ทาย 'ไม่ตาย' ทุกครั้งจะได้ Accuracy 88% ฟรีๆ โดยไม่เรียนรู้อะไร
→ ต้องดู Recall ของ class 1 (ตาย) เป็นหลัก ไม่ใช่ Accuracy รวม


## 3. Baseline: Logistic Regression (แสดงปัญหา Imbalance)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

from data_prep import FEATURES, CATEGORICAL_COLS

X = df[FEATURES]
y = df['target']
X_encoded = pd.get_dummies(X, columns=CATEGORICAL_COLS)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}, Features: {X_encoded.shape[1]}")

Train: (121422, 248), Test: (30356, 248), Features: 248


In [4]:
# Baseline: LR ธรรมดา (ไม่แก้ imbalance)
lr_base = LogisticRegression(max_iter=1000)
lr_base.fit(X_train, y_train)
y_pred_base = lr_base.predict(X_test)

print("=== Baseline LR (ไม่แก้ imbalance) ===")
print(classification_report(y_test, y_pred_base))
print("→ Recall class 1 (ตาย) = 0.05 : พลาดจุดเสี่ยง 95% ใช้งานจริงไม่ได้")

=== Baseline LR (ไม่แก้ imbalance) ===
              precision    recall  f1-score   support

           0       0.89      0.99      0.94     26817
           1       0.54      0.07      0.13      3539

    accuracy                           0.88     30356
   macro avg       0.72      0.53      0.53     30356
weighted avg       0.85      0.88      0.84     30356

→ Recall class 1 (ตาย) = 0.05 : พลาดจุดเสี่ยง 95% ใช้งานจริงไม่ได้


In [5]:
# แก้ด้วย class_weight='balanced'
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_balanced.fit(X_train, y_train)
y_pred_balanced = lr_balanced.predict(X_test)

print("=== LR + class_weight=balanced ===")
print(classification_report(y_test, y_pred_balanced))
print("→ Recall class 1 กระโดดจาก 0.05 → 0.63 เพราะโมเดลให้น้ำหนักกับ class ส่วนน้อยมากขึ้น")

=== LR + class_weight=balanced ===
              precision    recall  f1-score   support

           0       0.94      0.75      0.83     26817
           1       0.25      0.64      0.36      3539

    accuracy                           0.73     30356
   macro avg       0.59      0.69      0.59     30356
weighted avg       0.86      0.73      0.78     30356

→ Recall class 1 กระโดดจาก 0.05 → 0.63 เพราะโมเดลให้น้ำหนักกับ class ส่วนน้อยมากขึ้น


## 4. Feature Importance Analysis

In [6]:
import pandas as pd

coef_df = pd.DataFrame({
    'feature': X_encoded.columns,
    'coefficient': lr_balanced.coef_[0]
}).sort_values('coefficient', ascending=False)

print("ปัจจัยที่เพิ่มความเสี่ยงตายมากที่สุด (Top 10):")
print(coef_df.head(10).to_string(index=False))

print("\nปัจจัยที่ลดความเสี่ยงตายมากที่สุด (Bottom 10):")
print(coef_df.tail(10).to_string(index=False))

ปัจจัยที่เพิ่มความเสี่ยงตายมากที่สุด (Top 10):
                                                 feature  coefficient
            cause_clean_ความประมาท/เร่งรีบ (วิ่งตัดหน้า)     3.141470
                                first_vehicle_คนเดินเท้า     2.010133
cause_clean_รถเสียไม่แสดงเครื่องหมายหรือสัญญาณไฟที่กำหนด     1.772541
                                 cause_clean_ขับรถย้อนศร     1.403643
                                   agency_กรมทางหลวงชนบท     1.235703
           road_characteristic_จุดกลับรถต่างระดับ+ที่ราบ     1.206900
         road_characteristic_ทางแยกรูปตัว T+บนช่วงลาดชัน     1.128355
                         cause_clean_แซงรถอย่างผิดกฎหมาย     0.945333
                road_characteristic_ทางคนเดินเท้า+ที่ราบ     0.916669
                    cause_clean_ขับรถคร่อมเส้นแบ่งทิศทาง     0.878191

ปัจจัยที่ลดความเสี่ยงตายมากที่สุด (Bottom 10):
                                 feature  coefficient
                         province_ชลบุรี    -0.895058
      cause_clean_มีกองวัสด

## 5. Final Model: XGBoost

เลือก XGBoost เพราะ:
- จัดการ imbalanced data ได้ดีกว่าผ่าน `scale_pos_weight`
- Feature importance ตีความได้ตรงกว่า Random Forest
- LR แพ้ RF ในเรื่อง Recall แต่เมื่อเทียบ XGBoost แล้วเห็นว่า XGBoost ดีที่สุดในชุดนี้

In [7]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale:.2f}  (สัดส่วน class 0/class 1 ใน train set)")

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# threshold 0.4 เลือกจากการวิเคราะห์ Precision-Recall trade-off
# ยอม Precision ต่ำ (เตือนเกิน) เพื่อแลกกับ Recall สูง (ไม่พลาดจุดเสี่ยง)
y_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.4).astype(int)

print("\n=== Final XGBoost (threshold=0.4) ===")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

scale_pos_weight: 7.58  (สัดส่วน class 0/class 1 ใน train set)

=== Final XGBoost (threshold=0.4) ===
[[16163 10654]
 [  813  2726]]
              precision    recall  f1-score   support

           0       0.95      0.60      0.74     26817
           1       0.20      0.77      0.32      3539

    accuracy                           0.62     30356
   macro avg       0.58      0.69      0.53     30356
weighted avg       0.86      0.62      0.69     30356



In [8]:
# Feature Importance
importance_df = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance Top 10:")
print(importance_df.head(10).to_string(index=False))
print()
print("Finding: first_vehicle_รถจักรยานยนต์ สำคัญสูงสุด (0.14)")
print("Finding: agency_กรมทางหลวงชนบท อันดับ 2 → ถนนชนบทเสี่ยงกว่าถนนหลัก")

Feature Importance Top 10:
                                     feature  importance
                 first_vehicle_รถจักรยานยนต์    0.111124
                       agency_กรมทางหลวงชนบท    0.065740
                      province_กรุงเทพมหานคร    0.060172
                             province_ชลบุรี    0.029453
                    first_vehicle_คนเดินเท้า    0.022243
               first_vehicle_รถสามล้อเครื่อง    0.021021
    cause_clean_คน/รถ/สัตว์ตัดหน้ากระชั้นชิด    0.020099
cause_clean_ความประมาท/เร่งรีบ (วิ่งตัดหน้า)    0.017684
                     first_vehicle_รถจักรยาน    0.016462
                          province_เชียงใหม่    0.016127

Finding: first_vehicle_รถจักรยานยนต์ สำคัญสูงสุด (0.14)
Finding: agency_กรมทางหลวงชนบท อันดับ 2 → ถนนชนบทเสี่ยงกว่าถนนหลัก


## 6. บันทึกโมเดล

In [9]:
import joblib

joblib.dump(xgb_model, 'accident_risk_model.pkl')
joblib.dump(X_encoded.columns.tolist(), 'model_columns.pkl')
print("บันทึกโมเดลเรียบร้อย")
print(f"Features: {len(X_encoded.columns)} คอลัมน์")

บันทึกโมเดลเรียบร้อย
Features: 248 คอลัมน์


## 7. AI Magic Search — Hybrid Semantic Search

ใช้ `intfloat/multilingual-e5-small` + synonym expansion + fuzzy lexical matching
เพื่อให้ค้นหาด้วยภาษาธรรมชาติได้ เช่น 'ถนนมืด' → 'แสงสว่างไม่เพียงพอ'

In [10]:
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz
from data_prep import SEARCH_SYNONYMS

search_model = SentenceTransformer('intfloat/multilingual-e5-small')

# สร้างวลี→หมวดหมู่ (ชื่อหมวด + synonym ทั้งหมด)
phrase_to_category = {}
for category, synonyms in SEARCH_SYNONYMS.items():
    phrase_to_category[category] = category
    for phrase in synonyms:
        phrase_to_category[phrase] = category
for cat in df['cause_clean'].unique():
    if cat not in phrase_to_category:
        phrase_to_category[cat] = cat

all_phrases = list(phrase_to_category.keys())
phrase_embeddings = search_model.encode([f"passage: {p}" for p in all_phrases])
print(f"วลีอ้างอิงทั้งหมด: {len(all_phrases)} วลี")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

วลีอ้างอิงทั้งหมด: 150 วลี


In [ ]:
def hybrid_search(query, top_k=3, semantic_weight=0.85):
    """Hybrid: semantic embedding + lexical fuzzy matching"""
    query_embedding = search_model.encode(f"query: {query}")
    semantic_scores = util.cos_sim(query_embedding, phrase_embeddings)[0]

    category_best = {}
    for i, phrase in enumerate(all_phrases):
        category = phrase_to_category[phrase]
        sem = float(semantic_scores[i])
        lex = fuzz.partial_ratio(query, phrase) / 100
        score = semantic_weight * sem + (1 - semantic_weight) * lex
        if category not in category_best or score > category_best[category]:
            category_best[category] = score

    return sorted(category_best.items(), key=lambda x: -x[1])[:top_k]

# ทดสอบเบื้องต้น
print(hybrid_search("ถนนมืดมองไม่เห็น"))
print(hybrid_search("คนขับหลับคาพวงมาลัย"))
print(hybrid_search("ฝนตกถนนลื่น"))

## 8. Evaluation: Systematic Test (25 คำถาม)

In [ ]:
EVAL_SET = [
    ("กดคันเร่งจนรถพุ่งเร็วเกินไป", "ขับรถเร็วเกินอัตรากำหนด"),
    ("ขับรถทั้งที่ตาปรือง่วงมาก", "หลับใน"),
    ("กินเหล้าเยอะแล้วยังขับรถกลับบ้าน", "เมาสุรา"),
    ("พิมพ์แชทตอบไลน์ระหว่างขับรถ", "ใช้โทรศัพท์เคลื่อนที่ขณะขับรถ"),
    ("รถกระบะบรรทุกของสูงจนล้นคันรถ", "บรรทุกเกินอัตรา"),
    ("ยางเก่ามากจนระเบิดกลางทาง", "ยางเสื่อมสภาพ/ยางแตก"),
    ("หน้าฝนถนนลื่นมาก", "ถนนลื่น (เนื่องจากสภาพอากาศ)"),
    ("ไม่มีไฟถนนเลยตอนกลางคืน", "แสงสว่างไม่เพียงพอ"),
    ("โค้งหักศอกมองไม่เห็นรถที่สวนมา", "ทางโค้งอันตราย"),
    ("วัวเดินตัดหน้ารถกะทันหันตอนกลางคืน", "คน/รถ/สัตว์ตัดหน้ากระชั้นชิด"),
    ("ขับสวนเลนผิดทาง", "ขับรถย้อนศร"),
    ("แซงตรงทางโค้งที่ห้ามแซง", "แซงรถอย่างผิดกฎหมาย"),
    ("เหยียบเบรกแล้วรถไม่หยุด", "ระบบห้ามล้อขัดข้อง/ระบบเบรกชำรุด"),
    ("นักท่องเที่ยวขับรถหลงทางแล้วเกิดอุบัติเหตุ", "ไม่คุ้นเคยเส้นทาง/ขับรถไม่ชำนาญ"),
    ("ขับจี้ท้ายรถคันหน้าตลอดทาง", "ขับรถตามกระชั้นชิด"),
    ("ปาดขวาแซงแล้วตัดเข้าเลนกะทันหัน", "เปลี่ยนช่องทางกะทันหัน"),
    ("มีเศษไม้ตกกลางถนนตอนกลางคืน", "มีสิ่งกีดขวางบนทางหลวง (วัสดุหรือสัตว์)"),
    ("ขับรถทางไกลจนเหนื่อยล้าเกินไป", "ความเมื่อยล้า"),
    ("ไม่หยุดให้รถทางเอกไปก่อนตรงทางแยก", "ไม่ยอมให้รถที่มีสิทธิ์ไปก่อน"),
    ("หัวใจวายเฉียบพลันขณะขับรถ", "โรคประจำตัว"),
    ("ขับผ่านสี่แยกทั้งที่ไฟยังแดงอยู่", "ฝ่าฝืนสัญญาณไฟ/เครื่องหมายจราจร"),
    ("รถดับกลางถนนเพราะเครื่องมีปัญหา", "เครื่องยนต์ขัดข้อง"),
    ("หักหลบแล้วรถหมุนควงสว่าน", "สูญเสียการควบคุม"),
    ("ขับอยู่เลนขวาสุดทั้งที่ต้องใช้เลนซ้าย", "ขับรถผิดช่องทาง"),
    ("มีกองทรายกองอยู่กลางถนน", "มีกองวัสดุ/สิ่งกีดขวาง"),
]
print(f"Test cases: {len(EVAL_SET)} คำถาม")

In [ ]:
def evaluate_search(eval_set, top_k=3):
    results = []
    for query, expected in eval_set:
        predictions = hybrid_search(query, top_k=top_k)
        predicted = [p[0] for p in predictions]
        results.append({
            'query': query, 'expected': expected,
            'top1_correct': predicted[0] == expected,
            'topk_correct': expected in predicted,
        })

    top1_acc = sum(r['top1_correct'] for r in results) / len(results)
    topk_acc = sum(r['topk_correct'] for r in results) / len(results)

    print(f"Top-1 accuracy: {top1_acc:.1%}")
    print(f"Top-{top_k} accuracy: {topk_acc:.1%}")
    print()
    misses = [r for r in results if not r['topk_correct']]
    if misses:
        print("=== พลาด (ไม่ติด top-k) ===")
        for r in misses:
            print(f"❌ '{r['query']}' → ต้องการ '{r['expected']}'")
    else:
        print("✅ ผ่านทุกข้อ!")
    return results

eval_results = evaluate_search(EVAL_SET)

In [ ]:
from data_prep import load_and_clean_data, SEARCH_SYNONYMS
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz

df = load_and_clean_data()
search_model = SentenceTransformer('intfloat/multilingual-e5-small')

phrase_to_category = {}
for category, synonyms in SEARCH_SYNONYMS.items():
    phrase_to_category[category] = category
    for phrase in synonyms:
        phrase_to_category[phrase] = category
for cat in df['cause_clean'].unique():
    if cat not in phrase_to_category:
        phrase_to_category[cat] = cat

all_phrases = list(phrase_to_category.keys())
phrase_embeddings = search_model.encode([f"passage: {p}" for p in all_phrases])

def hybrid_search(query, top_k=3, semantic_weight=0.85):
    query_embedding = search_model.encode(f"query: {query}")
    semantic_scores = util.cos_sim(query_embedding, phrase_embeddings)[0]
    category_best = {}
    for i, phrase in enumerate(all_phrases):
        category = phrase_to_category[phrase]
        sem = float(semantic_scores[i])
        lex = fuzz.partial_ratio(query, phrase) / 100
        score = semantic_weight * sem + (1 - semantic_weight) * lex
        if category not in category_best or score > category_best[category]:
            category_best[category] = score
    return sorted(category_best.items(), key=lambda x: -x[1])[:top_k]

print(hybrid_search("หักหลบแล้วรถหมุนควงสว่าน"))